# 4 — Evaluate: the point head against three box systems, one scorer

Five prediction systems, the same 3,127 CFD Brackish val frames, the same box ground truth, and
**one** scoring function — Liam's `match_image`, copied verbatim from
`examples/WheatHead/notebooks/4_evaluate.ipynb` into
`examples/FishDetection/scripts/point_in_box.py`. Every system arrives here as a CVAT 1.1 points
XML written by `3_inference`, so nothing about a system's original output format can flatter or
penalise it at scoring time.

The experiment, stated once:

> A parked branch trained a **box** head on these frames (AP50 0.674 vs the released
> RF-DETR-Nano's 0.790) and, rescored as points, reached point-in-box F1 0.758 (best) / 0.746
> (last) against Nano's 0.782. This branch trains the repo's **native point head** — the same
> frozen trunk and the same fusion decoder, minus the `wh`/`off` geometry branch — on the same
> frames. **Does dropping that branch cost the heat branch anything, or free capacity?**

Outputs: `results/points/results.csv`, `results_summary.json`,
`figures/{accuracy_vs_threshold,per_clip_accuracy,accuracy_by_count_bucket,spot_checks_*}.png`.
No GPU needed — this notebook reads predictions, it does not make them.

## The honesty rules — read before any number below

These are the constraints this experiment is reported under. They were fixed before the
numbers, and they are what stops a single-seed read from being written up as a claim.

1. **Never compare against the CFD README's AP.** The only admissible baseline numbers are the
   ones re-scored here, on these frames, through this scorer. A published headline computed on a
   different split with a different scorer is not a comparison.
2. **RF-DETR-Nano very likely saw these val frames.** It was trained on all of CFD. Its row is
   biased **in its own favour**, which makes the asymmetry explicit: if we lose to it, the gap is
   *ambiguous* — it may be memorisation, not capability; if we beat it, the win is
   **conservative**. Its row carries a ⚠️ everywhere it appears.
3. **Report trainable parameters AND measured FLOPs.** Never "3.4 M vs 30 M". The frozen
   ConvNeXt-B trunk runs on every forward pass and dominates the arithmetic; parameters alone
   overstate how cheap this head is at inference.
4. **One seed is a read, not a claim.** One seed, one source, one split, one architecture change.
   Nothing here establishes anything; it tells us where to point the next run.
5. **"Freed capacity" is two mechanisms, and this experiment cannot separate them.** Removing the
   geometry branch removes ≈ 0.22 M parameters **and** two loss terms that were pulling on the
   *shared* fusion trunk. If the point head comes out ahead, this design cannot say which of
   those did it. Saying "freed capacity" as though it were the parameter count alone would be a
   claim the data does not support.
6. **"Indistinguishable at one seed" is a legitimate outcome.** The box run's own epoch-to-epoch
   AP75 swung by ±0.08 on this val split. A difference smaller than that kind of noise is a
   *null*, and a null is the honest answer — not a reason to hunt for a metric that separates
   the two.

## 1 · Paths, code, data

Same paths as `3_inference`. No model is constructed in this notebook, so the gated DINOv3
backbone is not needed and is not fetched; parameter counts and FLOPs are read back out of
`flops_params.json`, which `3_inference` measured on the GPU.

The bbox root (`DATA`) is still required: it holds the **box ground truth** and the per-image
`cfd_sequence` the per-clip breakdown groups by.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import importlib, json, os, sys
from pathlib import Path

DRIVE  = '/content/drive/MyDrive/frozen-trunk-detection'
REPO   = '/content/crop-counter'
DATA   = '/content/data/brackish'          # COCO bbox subset  -> the GT boxes
POINTS = '/content/data/brackish_points'   # COCO keypoints root + cfd_id_map.json
CFD    = '/content/cfd'
RUN    = f'{DRIVE}/runs/brackish_points_s0'
RES    = f'{DRIVE}/results/points'
FIGS   = f'{RES}/figures'
for sub in (RES, FIGS, f'{RES}/viz', CFD):
    os.makedirs(sub, exist_ok=True)
print('results dir:', RES, '->', sorted(os.listdir(RES)))

%cd /content
!rm -rf crop-counter
!git clone --branch poc/fish-points --depth 1 https://github.com/InsightML/crop-counter.git
%cd /content/crop-counter
!git log --oneline -3
!pip install -q -e ".[dev,portal,cfd]"
for path in ('/content/crop-counter/src', '/content/crop-counter/examples/FishDetection/scripts'):
    if path not in sys.path:
        sys.path.insert(0, path)
importlib.invalidate_caches()

import numpy as np
import nb_helpers as nbh
import point_in_box as pib
print('nb_helpers ', nbh.__file__)
print('point_in_box', pib.__file__)

In [ ]:
# Idempotent, and present in every notebook of this example: Colab attaches each notebook to its
# own runtime, so the frames 1_reformat fetched are not here unless this notebook shares that
# session. Only the bbox root's annotations are strictly needed below, but the spot-check figures
# want the pixels too, so the whole slice is restored.
have_points = os.path.exists(f'{POINTS}/val/annotations.json')
have_images = os.path.isdir(f'{DATA}/val/images') and len(os.listdir(f'{DATA}/val/images')) > 0
if not (have_points and have_images):
    META = f'{CFD}/community_fish_detection_dataset.json.zip'
    if not os.path.exists(META):
        !wget -q -O {META} https://lilawildlife.blob.core.windows.net/lila-wildlife/community-fish-detection-dataset/community_fish_detection_dataset.json.zip
    !python -m cropcounter.cfd subset --metadata {META} --out {DATA} --sources brackish_dataset --train-cap 100000 --val-cap 100000 --seed 0 --no-progress
    !python -m cropcounter.cfd fetch  --subset {DATA} --max-side 1024 --workers 32 --mirror gcs --no-progress
    !python -m cropcounter.cfd points --subset {DATA} --out {POINTS}
print('val image files:', len(os.listdir(f'{DATA}/val/images')))

## 2 · One key space — the GT and all five systems keyed by file name

Three id spaces exist in this experiment and conflating any two of them silently changes the
score:

| Space | Where it comes from |
| :-- | :-- |
| CFD **string** id (`brackish_dataset_…jpg`) | the bbox subset, and therefore the box-head / RF-DETR COCO results |
| points-root **int** id (1, 2, 3 …) | `cfd points` renumbered them, because `parse_coco_keypoints` does `int(image["id"])` |
| **file name** | what CVAT XMLs carry, and what `parse_pred_points` returns as its keys |

`3_inference` already resolved the first two into the third when it wrote the comparator XMLs
(through `cfd_image_id` / `cfd_id_map.json`), so everything here only has to re-key the GT the
same way — and then assert that all six mappings cover *exactly* the same 3,127 frames. An image
present in the GT but missing from a system is scored as zero predictions (that is what makes
Brackish's 1,740 empty frames punish false positives); an image present in a system but missing
from the GT would be a join bug, so it asserts instead.

In [ ]:
EXPECT_N_IMAGES = 3127
EXPECT_N_BOXES = 1965
EXPECT_N_EMPTY = 1740

bbox_doc = json.load(open(f'{DATA}/val/annotations.json'))
id_to_name, name_to_id = nbh.name_maps(bbox_doc)
gt = nbh.rekey_by_name(pib.coco_boxes_by_image(f'{DATA}/val/annotations.json'), id_to_name)

n_boxes = sum(len(b) for b in gt.values())
n_empty = sum(1 for b in gt.values() if not len(b))
print(f'GT: {len(gt)} frames | {n_boxes} boxes | {n_empty} empty frames '
      f'({n_empty / len(gt):.1%})')
assert len(gt) == EXPECT_N_IMAGES, f'{len(gt)} GT frames, expected {EXPECT_N_IMAGES}'
assert n_boxes == EXPECT_N_BOXES and n_empty == EXPECT_N_EMPTY

# Order matters: it is the row order of every table and the series order of every figure.
SYSTEMS = {
    'point head last':         f'{RES}/pred_points_last.xml',
    'point head best':         f'{RES}/pred_points_best.xml',
    'box head last (centres)': f'{RES}/pred_boxhead_last_centres.xml',
    'box head best (centres)': f'{RES}/pred_boxhead_best_centres.xml',
    'RF-DETR-Nano (centres)':  f'{RES}/pred_rfdetr_nano_centres.xml',
}
POINT_ROWS = ('point head last', 'point head best')

preds = {}
for label, path in SYSTEMS.items():
    preds[label] = pib.parse_pred_points(path)
    extra = set(preds[label]) - set(gt)
    assert not extra, f'{label}: {len(extra)} predicted frames are not in the GT'
    assert len(preds[label]) == EXPECT_N_IMAGES, f'{label}: {len(preds[label])} frames'
    total = sum(len(v) for v in preds[label].values())
    print(f'{label:<26} {len(preds[label])} frames | {total:>7} points at the floor tau 0.05')

## 3 · Threshold sweep, and the cross-check that the join is right

`sweep_thresholds` scores every system at τ = 0.05 … 0.95 (step 0.05). For each we record the
best-F1 threshold and the argmin-count-MAE threshold — they need not agree, and where they
disagree the report says so rather than quoting whichever flatters.

For the **point head** the headline operating point is the τ `2_training` calibrated, read from
`tau_calibration.json` (Liam's convention: the threshold is chosen on val during training, not
picked post hoc from the table it is about to fill in). The best-F1 τ is shown beside it as the
sanity check — if the calibrated τ sits far from the F1 optimum, calibration and evaluation
disagree about what "good" means and that is worth knowing.

**The cross-check.** The three comparator rows were scored on exactly this ground truth once
before, by `scripts/rescore_boxes_as_points.py`, straight from the COCO results — no CVAT round
trip in between. Those numbers are committed in `docs/free_first_step.json`. Reproducing them
here is the proof that `3_inference`'s id join and 0.05 score floor changed nothing. It is a
**tolerance, not an equality**: CVAT stores confidence as a whole percent, so a detection scoring
0.3499 is written as `35` and reads back as 0.35, landing on the other side of the τ = 0.35
threshold. Anything beyond ±0.02 F1 is a bug, not rounding.

In [ ]:
from tqdm.auto import tqdm

sweeps, best_f1, best_mae = {}, {}, {}
for label in tqdm(SYSTEMS, desc='sweeping'):
    sweeps[label] = pib.sweep_thresholds(preds[label], gt, nbh.SWEEP_THRESHOLDS)
    best_f1[label] = nbh.best_by(sweeps[label], 'f1', largest=True)
    best_mae[label] = nbh.best_by(sweeps[label], 'count_mae', largest=False)

print(f"{'system':<26} {'bestF1 tau':>10} {'F1':>7} {'minMAE tau':>11} {'MAE':>7}")
for label in SYSTEMS:
    f, m = best_f1[label], best_mae[label]
    flag = '' if abs(f['conf_thr'] - m['conf_thr']) < 1e-9 else '   <- disagree'
    print(f"{label:<26} {f['conf_thr']:>10.2f} {f['f1']:>7.4f} "
          f"{m['conf_thr']:>11.2f} {m['count_mae']:>7.3f}{flag}")

In [ ]:
# The calibrated operating point for the point head. NMS 1.5 is the run's own config and the one
# the results table uses; the notebook also wrote NMS-5 XMLs, which are not tabulated.
HEADLINE_NMS = 1.5
tau_cal, tau_notes = nbh.read_tau_calibration(
    f'{RES}/tau_calibration.json', checkpoints=('best', 'last'),
    nms_radii=(HEADLINE_NMS, 5.0), default_tau=0.3,
)
for line in tau_notes:
    print(line)

# tau per row: the calibrated one for the point head, the best-F1 one for the comparators (which
# have no calibration of their own -- they are other people's checkpoints, read at their best).
TAU = {label: float(best_f1[label]['conf_thr']) for label in SYSTEMS}
TAU_SOURCE = {label: 'best-F1 tau (this sweep)' for label in SYSTEMS}
for label in POINT_ROWS:
    TAU[label] = float(tau_cal[label.split()[-1]][HEADLINE_NMS])
    TAU_SOURCE[label] = 'calibrated in 2_training'

op, rows_at_tau = {}, {}
for label in SYSTEMS:
    op[label], rows_at_tau[label] = pib.score_dataset(preds[label], gt, TAU[label])
print('\noperating points:', {k: round(v, 2) for k, v in TAU.items()})

In [ ]:
# Cross-check against the committed free-first-step numbers (scored from the raw COCO results,
# with no CVAT round trip). A gap beyond CVAT's whole-percent rounding is a join bug.
FFS_PATH = 'examples/FishDetection/notebooks/docs/free_first_step.json'
FFS_LABEL = {'box head best (centres)': 'boxhead_best',
             'box head last (centres)': 'boxhead_last',
             'RF-DETR-Nano (centres)': 'rfdetr_nano'}
TOLERANCE = 0.02

published = {s['label']: s['best_f1'] for s in json.load(open(FFS_PATH))['systems']}
print(f"{'system':<26} {'ours F1':>8} {'published':>10} {'delta':>8}  tau ours/published")
worst = 0.0
for label, key in FFS_LABEL.items():
    ours, theirs = best_f1[label], published[key]
    delta = ours['f1'] - theirs['f1']
    worst = max(worst, abs(delta))
    print(f"{label:<26} {ours['f1']:>8.4f} {theirs['f1']:>10.4f} {delta:>+8.4f}  "
          f"{ours['conf_thr']:.2f} / {theirs['conf_thr']:.2f}")
assert worst <= TOLERANCE, (
    f'comparator F1 differs from docs/free_first_step.json by {worst:.4f} > {TOLERANCE}: '
    'the id join or the 0.05 score floor in 3_inference is wrong, not CVAT rounding'
)
print(f'\nlargest deviation {worst:.4f} <= {TOLERANCE} -> the join and the floor are sound; '
      'the residual is CVAT storing confidence as a whole percent.')

## 4 · The like-for-like table

Five systems, one scorer, one ground truth, one set of frames. Each row is reported **at its own
τ** (the column says which and the footnote says where it came from) — there is no single global
threshold in this table, and pretending otherwise would hand the win to whichever system happens
to like 0.35.

Read the F1 column against the box anchors and rule 6: a difference smaller than the box run's
own ±0.08 AP75 epoch noise is a null.

In [ ]:
flops = json.load(open(f'{RES}/flops_params.json'))
POINT_TRAINABLE_M = float(flops['trainable_params_M'])
POINT_GFLOPS = float(flops['gflops']['1024x576'])
# From the box memo (docs/report.md on the parked poc/detection-head branch), so that the cost
# columns are comparable: the box head at the SAME 1024x576 input, and RF-DETR-Nano at 640^2.
BOX_TRAINABLE_M, BOX_GFLOPS = 3.58, 477.0
NANO_TRAINABLE_M, NANO_GFLOPS = 30.0, 97.3

COST = {
    'point head last':         (POINT_TRAINABLE_M, POINT_GFLOPS),
    'point head best':         (POINT_TRAINABLE_M, POINT_GFLOPS),
    'box head last (centres)': (BOX_TRAINABLE_M, BOX_GFLOPS),
    'box head best (centres)': (BOX_TRAINABLE_M, BOX_GFLOPS),
    'RF-DETR-Nano (centres)':  (NANO_TRAINABLE_M, NANO_GFLOPS),
}
DISPLAY = {label: (label + ' ⚠️' if label.startswith('RF-DETR') else label)
           for label in SYSTEMS}

COLUMNS = [
    ('system', 'System', None),
    ('trainable_M', 'trainable M', '.2f'),
    ('gflops', 'GFLOPs', '.1f'),
    ('tau', 'τ', '.2f'),
    ('count_mae', 'count MAE', '.3f'),
    ('count_bias', 'bias', '+.3f'),
    ('precision', 'P', '.4f'),
    ('recall', 'R', '.4f'),
    ('f1', 'point-in-box F1', '.4f'),
    ('mean_accuracy', 'mean per-image acc', '.4f'),
]
EXTRA = ['row_kind', 'tau_source', 'count_rmse', 'tp', 'fp', 'fn', 'n_pred', 'n_gt',
         'n_images', 'gflops_shape']


def build_row(label, summary, tau_value, kind, tau_source):
    trainable, gflops_value = COST[label]
    return {
        'system': DISPLAY[label] + ('' if kind == 'headline' else ' @ best-F1 τ'),
        'trainable_M': trainable, 'gflops': gflops_value,
        'gflops_shape': '1024x576' if not label.startswith('RF-DETR') else '640x640',
        'tau': float(tau_value), 'row_kind': kind, 'tau_source': tau_source,
        **{k: summary[k] for k in ('count_mae', 'count_rmse', 'count_bias', 'precision',
                                   'recall', 'f1', 'mean_accuracy', 'tp', 'fp', 'fn',
                                   'n_pred', 'n_gt', 'n_images')},
    }


table = [build_row(label, op[label], TAU[label], 'headline', TAU_SOURCE[label])
         for label in SYSTEMS]
# The point head's best-F1 operating point, shown beside the calibrated headline (rule: the
# headline is the calibrated tau; this is the sanity check, not a second claim).
reference = [build_row(label, best_f1[label], best_f1[label]['conf_thr'],
                       'point_head_best_f1_reference', 'best-F1 tau (this sweep)')
             for label in POINT_ROWS]

FOOTNOTES = [
    'τ is **per row**: ' + ' · '.join(
        f'{DISPLAY[label]} τ={TAU[label]:.2f} ({TAU_SOURCE[label]})' for label in SYSTEMS),
    '',
    '- Scorer: point-in-box, greedy one-to-one, predictions ranked by confidence '
    '(`point_in_box.match_image`, copied verbatim from the WheatHead notebook). '
    '`mean per-image acc` is `tp/(tp+fp+fn)` averaged over frames, with an empty frame and no '
    'predictions counted as 1.0.',
    '- `trainable M` / `GFLOPs`: point-head rows measured in `3_inference` '
    f'({POINT_GFLOPS:.1f} GFLOPs at 1024×576); box-head and RF-DETR-Nano rows **from the '
    'box memo** (`docs/report.md`, parked `poc/detection-head`) — '
    f'{BOX_TRAINABLE_M:.2f} M / {BOX_GFLOPS:.0f} GFLOPs at the same 1024×576, and '
    f'≈{NANO_TRAINABLE_M:.0f} M / {NANO_GFLOPS} GFLOPs at 640².',
    '- ⚠️ RF-DETR-Nano was trained on all of CFD and very likely saw these val frames: '
    'its row is biased **in its favour**, so a loss to it is ambiguous and a win over it is '
    'conservative.',
    '- One seed, one source, one split. A difference below the box run’s own '
    '±0.08 AP75 epoch-to-epoch swing is a null.',
]

print(nbh.markdown_table(table, COLUMNS, FOOTNOTES))
print('\npoint head at its own best-F1 τ (the calibration sanity check, not the headline):')
print(nbh.markdown_table(reference, COLUMNS))

## 5 · Figure 11 analogue — accuracy vs confidence threshold

Mean per-image accuracy against τ, all five systems on one axis, with the point head's
calibrated operating points marked. This is the figure the interpretation turns on:

* **Same curve** (within the noise in rule 6) ⇒ the `wh`/`off` branch cost the heat branch
  nothing. The geometry head was free; removing it frees nothing either.
* **Separated curves** ⇒ a real effect at one seed. Which direction matters, and rule 5 still
  applies: 0.22 M parameters *and* two loss terms came off the shared trunk together, and this
  design cannot say which did the work.
* **Curves that cross** ⇒ the two heads have different score calibrations, not different
  abilities, and the headline should be read at each one's own τ (which is what the table does).

In [ ]:
from IPython.display import Image as IPImage, display

fig11 = nbh.plot_accuracy_vs_threshold(
    sweeps, Path(f'{FIGS}/accuracy_vs_threshold.png'),
    marked={label: TAU[label] for label in POINT_ROWS},
    title=('Mean per-image accuracy vs confidence threshold — CFD Brackish val '
           f'({len(gt):,} frames, {sum(len(b) for b in gt.values()):,} GT boxes)'),
)
display(IPImage(str(fig11)))

peak = {label: nbh.best_by(sweeps[label], 'mean_accuracy', largest=True) for label in SYSTEMS}
print('peak mean accuracy per system (and the tau it peaks at):')
for label in SYSTEMS:
    print(f"  {label:<26} {peak[label]['mean_accuracy']:.4f} @ tau {peak[label]['conf_thr']:.2f}")
spread = max(p['mean_accuracy'] for p in peak.values()) - min(p['mean_accuracy'] for p in peak.values())
print(f'\nspread across all five systems at their own peaks: {spread:.4f}')

## 6 · "WDA" — why it collapses here, and what replaces it

The wheat report's headline is **weighted domain accuracy**: the mean over *domains* of the mean
per-image accuracy inside each domain. Said plainly:

> **On Brackish, WDA is exactly the mean per-image accuracy.** Brackish is ONE domain — one
> fixed camera in one Danish harbour — so the outer average is over a single value and adds
> nothing. Quoting a "WDA" here would imply a robustness measurement that was not made.

What is worth having instead is a *variance* under the single number, so two breakdowns:

1. **Per clip** (`cfd_sequence`) — the val split's clips. Each is a different time of day,
   turbidity and fish traffic, which is the nearest thing to a domain axis this source has.
2. **By GT count** (0 / 1 / 2 / 3–5 / 6+) — where a counter actually differs from a detector.
   The 0 bucket is the majority of Brackish and scores only false positives; the 6+ bucket is
   where a one-peak-per-cell head is most likely to lose fish.

Every system is shown at **its own** τ (the same operating points as the table).

In [ ]:
seq_by_name = nbh.sequence_by_name(bbox_doc)
missing = set(gt) - set(seq_by_name)
assert not missing, f'{len(missing)} val frames have no cfd_sequence'

per_clip = {label: nbh.group_accuracy(rows_at_tau[label], lambda r: seq_by_name[r['image_id']])
            for label in SYSTEMS}
clips = sorted(per_clip['point head best'], key=lambda c: -per_clip['point head best'][c]['n_images'])
print(f'{len(clips)} val clips | frames per clip '
      f'{min(per_clip["point head best"][c]["n_images"] for c in clips)}'
      f'-{max(per_clip["point head best"][c]["n_images"] for c in clips)}')

fig_clip = nbh.plot_grouped_bars(
    [c[-26:] for c in clips],
    {label: [per_clip[label][c]['mean_accuracy'] for c in clips] for label in SYSTEMS},
    Path(f'{FIGS}/per_clip_accuracy.png'),
    title=(f'Mean per-image accuracy by val clip ({len(clips)} clips) — each system at its '
           'own τ. WDA would average these columns; Brackish is one domain, so the '
           'spread ACROSS them is the information.'),
    xlabel='cfd_sequence (val clip)',
    annotations=[f"n={per_clip['point head best'][c]['n_images']}" for c in clips],
    hline=(op['point head best']['mean_accuracy'], 'point head best — overall mean'),
    figsize=(16.0, 5.6),
)
display(IPImage(str(fig_clip)))
for label in SYSTEMS:
    values = np.array([per_clip[label][c]['mean_accuracy'] for c in clips])
    print(f'{label:<26} per-clip mean {values.mean():.4f} | sd {values.std():.4f} | '
          f'min {values.min():.4f} ({clips[int(values.argmin())][-26:]}) | max {values.max():.4f}')

In [ ]:
by_bucket = {label: nbh.group_accuracy(rows_at_tau[label], lambda r: nbh.count_bucket(r['gt_count']))
             for label in SYSTEMS}
buckets = nbh.bucket_order(by_bucket['point head best'])

fig_bucket = nbh.plot_grouped_bars(
    buckets,
    {label: [by_bucket[label][b]['mean_accuracy'] for b in buckets] for label in SYSTEMS},
    Path(f'{FIGS}/accuracy_by_count_bucket.png'),
    title=('Mean per-image accuracy by ground-truth count — each system at its own τ. '
           'The 0 bucket scores false positives only; 6+ is where a one-peak-per-cell head '
           'should struggle first.'),
    xlabel='GT fish in frame',
    annotations=[f"n={by_bucket['point head best'][b]['n_images']}" for b in buckets],
    figsize=(11.0, 5.2),
)
display(IPImage(str(fig_bucket)))
header = f"{'system':<26}" + ''.join(f'{b:>9}' for b in buckets)
print(header + '\n' + '-' * len(header))
for label in SYSTEMS:
    print(f'{label:<26}' + ''.join(f"{by_bucket[label][b]['mean_accuracy']:>9.4f}" for b in buckets))
print(f"\nframes per bucket: " + ', '.join(
    f"{b}: {by_bucket['point head best'][b]['n_images']}" for b in buckets))

## 7 · Spot checks — the frames where the choice of head changes the answer

Six frames, chosen deterministically: the three densest, then the three where the systems'
kept-prediction counts disagree most at their own τ. Those are the frames a table cannot
explain — a shared F1 hides whether two heads are right about the same fish.

GT boxes **green** · point head **red** · box-head centres **orange** · RF-DETR-Nano **blue**.
Two figures, one per checkpoint pair, so `best` and `last` can be compared frame for frame.

In [ ]:
SPOT = {
    'best': ('point head best', 'box head best (centres)', 'RF-DETR-Nano (centres)'),
    'last': ('point head last', 'box head last (centres)', 'RF-DETR-Nano (centres)'),
}
picks = nbh.pick_informative_frames(gt, preds, TAU, n=6, n_dense=3)
print('spot-check frames:')
for name in picks:
    counts = {label: int((preds[label][name][:, 2] >= TAU[label]).sum()) for label in SYSTEMS}
    print(f'  GT {len(gt[name]):>2} | ' + ' '.join(f'{k.split()[0]}:{v}' for k, v in counts.items())
          + f' | {name.split("_jpg.rf.")[0][-40:]}')

for ck, labels in SPOT.items():
    frames = []
    for name in picks:
        kept = {label: preds[label][name][preds[label][name][:, 2] >= TAU[label]]
                for label in labels}
        frames.append({
            'image_path': f'{DATA}/val/images/{name}',
            'boxes': gt[name],
            'points': [(label, kept[label][:, :2]) for label in labels],
            'title': (f'{name.split("_jpg.rf.")[0][-34:]}\nGT {len(gt[name])} · '
                      + ' · '.join(f'{label.split(" (")[0]} {len(kept[label])}'
                                        for label in labels)),
        })
    path = nbh.save_overlay_grid(
        frames, Path(f'{FIGS}/spot_checks_{ck}.png'), ncols=2,
        suptitle=(f'Spot checks ({ck} checkpoints) — GT boxes green · point head red '
                  f'· box-head centres orange · RF-DETR-Nano blue ⚠️; '
                  'each system at its own τ'),
    )
    display(IPImage(str(path)))

## 8 · Write the results, and the table once more

`results.csv` holds the five headline rows plus the point head's two best-F1 reference rows
(the `row_kind` column separates them). `results_summary.json` carries the same numbers plus the
full sweeps, the per-clip and per-bucket breakdowns, the free-first-step cross-check and the
operating points — everything the memo needs without re-running a notebook.

In [ ]:
csv_path = nbh.write_results_csv(
    table + reference, Path(f'{RES}/results.csv'),
    columns=[(k, h, s) for k, h, s in COLUMNS] + [(k, k, None) for k in EXTRA],
)

summary = {
    'experiment': ('cropcounter native point head vs a box head and RF-DETR-Nano, '
                   'point-in-box scored on CFD Brackish val'),
    'run': RUN,
    'val': {'n_images': len(gt), 'n_gt_boxes': int(sum(len(b) for b in gt.values())),
            'n_empty_images': int(sum(1 for b in gt.values() if not len(b))),
            'n_clips': len(clips)},
    'scorer': ('examples/FishDetection/scripts/point_in_box.py — match_image copied '
               'verbatim from examples/WheatHead/notebooks/4_evaluate.ipynb'),
    'thresholds': list(nbh.SWEEP_THRESHOLDS),
    'tau': TAU, 'tau_source': TAU_SOURCE, 'tau_notes': tau_notes,
    'flops_params': flops,
    'cost_from_box_memo': {'box_head': {'trainable_M': BOX_TRAINABLE_M, 'gflops': BOX_GFLOPS,
                                        'shape': '1024x576'},
                           'rfdetr_nano': {'trainable_M': NANO_TRAINABLE_M,
                                           'gflops': NANO_GFLOPS, 'shape': '640x640'}},
    'rows': table + reference,
    'operating_points': op,
    'best_f1': best_f1, 'best_count_mae': best_mae, 'peak_mean_accuracy': peak,
    'sweeps': sweeps,
    'per_clip_accuracy': per_clip,
    'accuracy_by_count_bucket': by_bucket,
    'free_first_step_crosscheck': {
        'source': FFS_PATH, 'tolerance': TOLERANCE, 'worst_abs_delta_f1': worst,
        'delta_f1': {label: float(best_f1[label]['f1'] - published[key]['f1'])
                     for label, key in FFS_LABEL.items()},
    },
    'spot_check_frames': list(picks),
    'honesty_rules': [
        'never compare against the CFD README AP',
        'RF-DETR-Nano likely saw these val frames: its row is biased in its favour, so our loss '
        'is ambiguous and our win is conservative',
        'trainable params AND measured FLOPs, never 3.4 M vs 30 M',
        'one seed is a read, not a claim',
        'freed capacity = 0.22 M params AND two loss terms off the shared fusion trunk; this '
        'experiment cannot separate the two mechanisms',
        'a difference below the box run’s own +-0.08 AP75 epoch swing is a null',
    ],
}
with open(f'{RES}/results_summary.json', 'w') as fh:
    json.dump(summary, fh, indent=1, default=float)

print(nbh.markdown_table(table, COLUMNS, FOOTNOTES))
print(f'\nwrote {csv_path}')
print(f'wrote {RES}/results_summary.json')
print('figures:', sorted(os.listdir(FIGS)))